In [2]:
# @title Simple Interactive S-Curve: The "Smash" vs. The "Unfold" { run: "auto" }

import matplotlib.pyplot as plt
from sklearn import datasets, decomposition, manifold
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

def simple_demo(view_angle=10, n_points=800):
    # 1. Generate S-Curve Data (A folded sheet of paper)
    X, color = datasets.make_s_curve(n_points, noise=0.0)

    # 2. PCA (Linear Flattening)
    pca = decomposition.PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # 3. Isomap (Non-Linear Unfolding)
    # This algorithm understands "walking along the paper"
    iso = manifold.Isomap(n_neighbors=10, n_components=2)
    X_iso = iso.fit_transform(X)

    # --- Plotting ---
    fig = plt.figure(figsize=(15, 5))

    # Plot A: The Reality (3D)
    ax1 = fig.add_subplot(131, projection='3d')
    ax1.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, cmap=plt.cm.Spectral, s=20)
    ax1.view_init(10, view_angle)
    ax1.set_title("1. Reality (3D S-Curve)\nNote the Fold", fontsize=11)

    # Plot B: PCA Failure
    ax2 = fig.add_subplot(132)
    ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=color, cmap=plt.cm.Spectral, s=20)
    ax2.set_title("2. PCA (The Mistake)\nIt smashes the top and bottom together.\n(Red touches Blue!)", fontsize=11, color='darkred')
    ax2.grid(True, alpha=0.3)

    # Plot C: The Solution
    ax3 = fig.add_subplot(133)
    ax3.scatter(X_iso[:, 0], X_iso[:, 1], c=color, cmap=plt.cm.Spectral, s=20)
    ax3.set_title("3. Isomap (The Solution)\nIt unfolds the paper perfectly.", fontsize=11, color='darkgreen')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Widget to rotate the 3D view to see the fold better
interact(simple_demo,
         view_angle=IntSlider(min=-180, max=180, step=5, value=-70, description='Rotate 3D:'),
         n_points=IntSlider(min=300, max=2000, step=100, value=800, description='Points:'));

interactive(children=(IntSlider(value=-70, description='Rotate 3D:', max=180, min=-180, step=5), IntSlider(val…

In [3]:
# @title Galería de Fallos de PCA: Esferas, Anillos y Hélices { run: "auto" }

import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets, decomposition, manifold
from ipywidgets import interact, Dropdown, IntSlider

def generate_shapes(shape, n_samples=1000, noise=0.05):
    """Generates complex 3D shapes."""
    if shape == 'Toroidal Helix':
        t = np.linspace(0, 4 * np.pi, n_samples)
        x = (2 + np.cos(8 * t)) * np.cos(t)
        y = (2 + np.cos(8 * t)) * np.sin(t)
        z = np.sin(8 * t)
        data = np.vstack((x, y, z)).T
        color = t  # Color by time/position along the wire
        title = "Hélice Toroidal (Bobina)"

    elif shape == 'Hollow Sphere':
        # Points on a sphere surface
        indices = np.arange(0, n_samples, dtype=float) + 0.5
        phi = np.arccos(1 - 2*indices/n_samples)
        theta = np.pi * (1 + 5**0.5) * indices
        x, y, z = np.cos(theta) * np.sin(phi), np.sin(theta) * np.sin(phi), np.cos(phi)
        data = np.vstack((x, y, z)).T * 5
        color = z  # Color by height (Z-axis)
        title = "Esfera Hueca (Cáscara)"

    elif shape == 'Linked Rings':
        # Two interlocking rings
        n = n_samples // 2
        t = np.linspace(0, 2*np.pi, n)

        # Ring 1 (Flat on XY)
        r1 = np.vstack((np.cos(t)*3 + 2, np.sin(t)*3, np.zeros(n))).T
        # Ring 2 (Standing on XZ, interlocking)
        r2 = np.vstack((np.cos(t)*3 - 2, np.zeros(n), np.sin(t)*3)).T

        data = np.vstack((r1, r2)) + np.random.normal(0, noise, (2*n, 3))
        color = np.concatenate((np.zeros(n), np.ones(n))) # Class 0 vs Class 1
        title = "Anillos Entrelazados (Linked Rings)"

    elif shape == 'Trefoil Knot':
        t = np.linspace(0, 4*np.pi, n_samples)
        x = np.sin(t) + 2 * np.sin(2 * t)
        y = np.cos(t) - 2 * np.cos(2 * t)
        z = -np.sin(3 * t)
        data = np.vstack((x, y, z)).T
        color = t
        title = "Nudo de Trébol (Trefoil Knot)"

    return data, color, title

def plot_shapes_demo(shape='Toroidal Helix', n_samples=1000, view_angle=45):
    X, color, shape_title = generate_shapes(shape, n_samples)

    # --- 1. PCA (Linear) ---
    pca = decomposition.PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # --- 2. t-SNE / Isomap (Non-Linear) ---
    # Usamos t-SNE porque es muy bueno separando clústeres complejos
    tsne = manifold.TSNE(n_components=2, init='pca', learning_rate='auto', perplexity=30)
    try:
        X_tsne = tsne.fit_transform(X)
    except:
        X_tsne = np.zeros((n_samples, 2)) # Fallback if error

    # --- Plotting ---
    fig = plt.figure(figsize=(18, 6))

    # 3D Original
    ax1 = fig.add_subplot(131, projection='3d')
    scatter1 = ax1.scatter(X[:, 0], X[:, 1], X[:, 2], c=color, cmap='jet', s=10, alpha=0.8)
    ax1.set_title(f"1. Original 3D: {shape_title}")
    ax1.view_init(20, view_angle)
    ax1.set_axis_off()

    # PCA
    ax2 = fig.add_subplot(132)
    ax2.scatter(X_pca[:, 0], X_pca[:, 1], c=color, cmap='jet', s=10, alpha=0.6)
    ax2.set_title("2. PCA (Proyección Lineal)\nNota cómo se solapan los colores", color='darkred')
    ax2.grid(True, alpha=0.3)

    # Non-Linear
    ax3 = fig.add_subplot(133)
    ax3.scatter(X_tsne[:, 0], X_tsne[:, 1], c=color, cmap='jet', s=10, alpha=0.6)
    ax3.set_title("3. t-SNE (Topología Preservada)\nSepara o desenrolla la estructura", color='darkgreen')
    ax3.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# Widgets
interact(plot_shapes_demo,
         shape=Dropdown(options=['Toroidal Helix', 'Hollow Sphere', 'Linked Rings', 'Trefoil Knot'], value='Linked Rings'),
         n_samples=IntSlider(min=500, max=2000, step=100, value=1000),
         view_angle=IntSlider(min=-180, max=180, step=10, value=45));

interactive(children=(Dropdown(description='shape', index=2, options=('Toroidal Helix', 'Hollow Sphere', 'Link…